# 04 · Pack the training shards (Milestone 3, decisions of 15 Nov 2026)

Writes the training and validation images of the CP1 run-5 manifest as their **harmonized
bytes** (the CP1 plan; COCO prepared to its DiffSeg30k size) into ~1 GB tar shards per source
under `Paper 1/cache/shards/`, with a **1-bit PNG edit mask** for every edited image (CP1 reading
rule: an AI-generated base's value read as 0) and `index.parquet` (one row per image, with
`mask_base`, `valid_crop` and the mask-rule version). The original dataset files are not touched.

Steps: Drive free space and the estimated shard size are printed first; packing **stops and asks**
if free space < estimate + 10 GB. Then the four CP1 sources are copied to the local disk, the
shards are written (resumable: re-run this notebook after a disconnect and finished shards are
skipped), and a seeded sample of every source is checked: packed bytes == a fresh harmonization.

**Result** (send to Claude): `Paper 1/runs/checkpoints/pack_shards.json`. CPU runtime.

In [ ]:
# Cell 1: mount Drive, get the code, install it
import os, subprocess, sys

DRY = os.environ.get('EVIGROUND_DRY_RUN') == '1'   # CI / local test: fake sources, temp folders
REPO_URL = 'github.com/DHANUSHKUMMARI/Paper1.git'
BRANCH = 'main'                                    # 'claude/nice-curie-3tzxzr' before the merge

if DRY:
    REPO = os.path.abspath('..' if os.path.basename(os.getcwd()) == 'notebooks' else '.')
else:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    REPO = '/content/Paper1'
    if not os.path.exists(REPO):
        try:
            tok = userdata.get('GITHUB_TOKEN')       # never printed
        except Exception:
            tok = None
        url = f'https://{tok}@{REPO_URL}' if tok else f'https://{REPO_URL}'
        r = subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, url, REPO],
                           capture_output=True, text=True)
        if r.returncode != 0:                        # stderr is not shown: it can contain the URL
            raise RuntimeError('git clone failed: check GITHUB_TOKEN (Secrets panel) and BRANCH')
        subprocess.run(['git', '-C', REPO, 'remote', 'set-url', 'origin', f'https://{REPO_URL}'],
                       check=True)
    else:
        subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO], check=True)

sys.path.insert(0, os.path.join(REPO, 'src'))
# Re-running in a runtime that already imported eviground would keep the OLD code in memory
# although git pulled the new one: drop it so the next import reads the files on disk.
for _name in [m for m in sys.modules if m == 'eviground' or m.startswith('eviground.')]:
    del sys.modules[_name]
print('repo:', REPO, '| branch:', BRANCH, '| dry run:', DRY)

In [ ]:
# Cell 2: settings, paths, Drive free space
import json, logging, os, time
logging.basicConfig(level=logging.INFO, force=True)
from eviground.env import drive_free_bytes, get_paths, session_info

WORKERS = os.cpu_count() or 2
T0 = time.time()
info = session_info()
paths = get_paths()
if DRY:   # a fresh fake Drive of its own
    import shutil, tempfile
    from pathlib import Path
    from eviground.env import Paths
    _dry = Path(tempfile.gettempdir()) / 'eviground_pack_dry'
    shutil.rmtree(_dry, ignore_errors=True)
    paths = Paths(root=_dry / 'drive' / 'Paper 1', local_data=_dry / 'data')
paths.ensure()
MANIFEST = paths.runs / 'checkpoints' / 'cp1' / 'manifest_cp1.csv.gz'
FREE = 10**12 if DRY else drive_free_bytes()
print('commit', info['git_commit'], '| workers', WORKERS, '| manifest', MANIFEST)
print('Drive free:', 'unknown' if FREE is None else f'{FREE / 1e9:.1f} GB')

In [ ]:
# Cell 3: the four CP1 sources on the local disk (dry run: a fake world and a fake CP1 run)
if DRY:
    from eviground.data.cp1 import CP1Config, run_cp1
    from eviground.data.download import ensure_data
    from eviground.data.fake_cp1 import cp1_overrides, make_cp1_world
    W = {'coco_images': 40}
    sources = make_cp1_world(paths.datasets, paths.root.parent / 'fake_remote', **W)
    ensure_data('cp1', paths=paths, sources=sources, stage_local=True,
                overrides=cp1_overrides(W['coco_images']))
    dry_cfg = CP1Config(workers=2, crops_per_class=60, blockiness_per_group=20, base_unclear_max=0.2,
                        diffseg_card_splits={'train': 18, 'validation': 12})
    run_cp1(paths, dry_cfg, out_dir=MANIFEST.parent, cache_dir=paths.cache / 'cp1', progress=None)
else:
    from eviground.data import cp1
    from eviground.data.download import stage
    assert MANIFEST.exists(), f'{MANIFEST} not found: it is written by notebook 01 (CP1 run 5)'
    for key in cp1.STAGED_KEYS:
        print(key, stage(key, paths=paths))

In [ ]:
# Cell 4: estimate, space check (stops and asks), pack. Re-run after a disconnect: it resumes.
from eviground.data import pack

SHARDS = paths.shards
summary = pack.pack(
    MANIFEST, SHARDS, free_bytes=FREE, workers=WORKERS,
    shard_bytes=200_000 if DRY else pack.SHARD_BYTES,
    sample_per_source=5 if DRY else 200, progress=print,
)
print('images', summary['images'], '| edit masks', summary['edit_masks'], '| shards',
      summary['shards'], f"| {summary['bytes'] / 1e9:.2f} GB written, estimate "
      f"{summary['estimate']['total_bytes'] / 1e9:.2f} GB")
for k, v in summary['by_source_split_label'].items():
    print(f'  {k}: {v}')

In [ ]:
# Cell 5: packed bytes == a fresh harmonization, for a seeded sample of every source; save
import datetime
from eviground.data import cp1

check = pack.verify_sample(SHARDS, cp1._read_csv_gz(MANIFEST), per_source=200, seed=42)
record = {
    'step': 'pack training shards',
    'utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
    'dry_run': DRY, 'branch': BRANCH, 'session': info,
    'drive_free_before_bytes': FREE,
    'wall_time_h': round((time.time() - T0) / 3600, 3),
    'index_sha256': pack.file_sha256(SHARDS / 'index.parquet'),
    'summary': summary, 'verify': check,
}
out = paths.runs / 'checkpoints' / 'pack_shards.json'
out.write_text(json.dumps(record, indent=1, default=str))
print('verify (packed == fresh harmonization):', {k: v for k, v in check.items()})
print('saved', out, '| wall time', round((time.time() - T0) / 60, 1), 'min')
assert check['all_equal'], 'packed bytes differ from a fresh harmonization: send the JSON to Claude'